# Betty: the cylinder of maximum volume

[Open in Colab](https://colab.research.google.com/github/gromicho/teaching/blob/main/foundations/optimization/betty-cylinder.ipynb) · [Open in Binder](https://mybinder.org/v2/gh/gromicho/teaching/main?urlpath=tree/foundations/optimization/betty-cylinder.ipynb)

By Joaquim Gromicho. Modernized from the original teaching notebook.

Choose a radius and height under a fixed surface-area budget. We first derive the model, then introduce the numerical solver and check its answer analytically.


# Betty's problem: the cylinder of maximum volume

$$
\begin{array}{rl}
\max    & \pi r^2 h                   \\
s.t.    & 2\pi r^2 + 2\pi r h \leq 12 \\
        & r \geq 0                    \\
        & h \geq 0  
\end{array}
$$

The objective is volume; the budget includes both circular ends. The feasible set includes degenerate zero-volume cylinders. A numerical local solve alone does not establish a global optimum.


In [ ]:
# Use installed packages, install only missing ones, without version pins.
# Load the shared teaching utilities from this checkout or a verified download.
from pathlib import Path
import hashlib
import sys
from urllib.request import urlopen

support_path = next((folder / 'support' for folder in [Path.cwd(), *Path.cwd().parents]
                     if (folder / 'support' / 'teaching_utils.py').is_file()), None)
if support_path is None:
    support_path = Path.cwd() / '.teaching-support'
    support_path.mkdir(exist_ok=True)
    helper = support_path / 'teaching_utils.py'
    expected = 'fbfa41e12709a01548e213abba976dad1e21d0abcfd798a2dfd669706cc152bb'
    if not helper.exists() or hashlib.sha256(helper.read_bytes()).hexdigest() != expected:
        url = 'https://raw.githubusercontent.com/gromicho/teaching/f3ad11b77cae7dd05315d314c7e72ee8516aaa3d/support/teaching_utils.py'
        content = urlopen(url, timeout=45).read()
        if hashlib.sha256(content).hexdigest() != expected:
            raise ValueError('Teaching helper version changed; reopen the current course notebook.')
        helper.write_bytes(content)
sys.path.insert(0, str(support_path))
from teaching_utils import ensure_packages

required_packages = {'sympy': 'sympy', 'numpy': 'numpy', 'matplotlib': 'matplotlib'}
ensure_packages(required_packages)


In [ ]:
import math
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
radius = sp.Symbol('r', nonnegative=True)
surface_budget = 12
height_on_boundary = (surface_budget - 2*sp.pi*radius**2)/(2*sp.pi*radius)
volume_on_boundary = sp.simplify(sp.pi*radius**2*height_on_boundary)
display(height_on_boundary, volume_on_boundary, sp.diff(volume_on_boundary, radius))
r_star = sp.sqrt(surface_budget/(6*sp.pi))
h_star = sp.simplify(height_on_boundary.subs(radius,r_star))
v_star = sp.simplify(volume_on_boundary.subs(radius,r_star))
display(r_star,h_star,v_star)


For positive radius, increasing height increases volume, so the surface constraint is tight at a nondegenerate optimum. Eliminating height gives a concave cubic on the feasible radius interval. Check the stationary point and the two limiting zero-volume endpoints; the optimum has height equal to the diameter.


In [ ]:
r_grid=np.linspace(0,math.sqrt(surface_budget/(2*math.pi)),200)
v_grid=surface_budget*r_grid/2-math.pi*r_grid**3
plt.plot(r_grid,v_grid,label='feasible boundary volume')
plt.scatter([float(r_star)],[float(v_star)],label='analytical optimum')
plt.xlabel('radius');plt.ylabel('volume');plt.legend();plt.show()


## Now build the two-variable Pyomo model
Pyomo and Ipopt enter here, after the geometry and symbolic calculation. The historical notebook used a remote solver before a local one. This edition makes an actual local solve and retains explicit solver discovery and installation at this stage.


In [ ]:
from teaching_utils import ensure_packages
required_packages = {'pyomo': 'pyomo'}
ensure_packages(required_packages)


In [ ]:
import pyomo.environ as pyo
from teaching_utils import available_pyomo_solvers, install_coin_solvers, make_solver, solve_checked
print('Before installation:',available_pyomo_solvers(['ipopt']))
install_coin_solvers()
print('After installation:',available_pyomo_solvers(['ipopt']))


In [ ]:
def cylinder_model(start=(0.7,1.5)):
    betty=pyo.ConcreteModel('Betty: maximum-volume cylinder')
    betty.r=pyo.Var(domain=pyo.NonNegativeReals,initialize=start[0])
    betty.h=pyo.Var(domain=pyo.NonNegativeReals,initialize=start[1])
    betty.volume=pyo.Objective(expr=math.pi*betty.r**2*betty.h,sense=pyo.maximize)
    betty.surface=pyo.Constraint(expr=2*math.pi*betty.r**2+2*math.pi*betty.r*betty.h<=surface_budget)
    return betty

solutions=[]
for start in [(0.7,1.5),(0.4,3.0),(1.0,0.5)]:
    betty=cylinder_model(start)
    result=solve_checked(betty,'ipopt')
    r,h,v=pyo.value(betty.r),pyo.value(betty.h),pyo.value(betty.volume)
    assert math.isclose(v,float(v_star),rel_tol=1e-6)
    assert math.isclose(h,2*r,rel_tol=1e-5)
    assert pyo.value(betty.surface.body)<=surface_budget+1e-5
    solutions.append((start,r,h,v,str(result.solver.termination_condition)))
solutions


## Investigate
Change the surface budget and repeat the symbolic and numerical checks. Explain why starting at radius zero is problematic: the volume gradient is zero there even though useful cylinders exist. Compare solver availability with evidence that a particular model was actually solved.
